In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import math
import random
import numpy as np

# ============================================================
# TRANSPORTATION SIMULATOR
# Three real-life supply/demand scenarios + VAM / Least Cost / MODI
# Requires: pip install numpy
# ============================================================

CITIES = ["Company 1", "Company 2", "Company 3", "Company 4"]
DEPOTS = [
    "Bank", "Hospital", "Store", "Telecom", "University",
    "Insurance", "Airline", "Factory", "Ministry", "Media"
]
COST = np.array([
    [1, 3, 9, 8, 5, 2, 2, 4, 12, 3],
    [4, 4, 14, 12, 9, 2, 7, 7, 16, 2],
    [18, 20, 8, 10, 13, 19, 20, 22, 4, 21],
    [13, 15, 4, 6, 8, 15, 16, 18, 1, 17]
], dtype=float)
SUPPLY = [40, 60, 50, 50]
SCENARIOS = [
    ("DEMAND EXCEEDS SUPPLY", [30, 30, 20, 20, 20, 20, 20, 20, 20, 20],
     "Demand is higher than available staff. A dummy source represents the shortage."),
    ("SUPPLY EXCEEDS DEMAND", [15, 15, 20, 20, 20, 20, 20, 20, 20, 10],
     "Available staff exceed requested demand. A dummy destination represents the surplus."),
    ("BALANCED OPERATION", [20, 20, 20, 20, 20, 20, 20, 20, 20, 20],
     "Available supply matches total demand. No dummy source or destination is needed.")
]
BG = "#101820"
PANEL = "#182630"
PANEL2 = "#213541"
TEXT = "#EDF4F7"
MUTED = "#9FB2BD"
CYAN = "#48D6D2"
ORANGE = "#FFB45B"
GREEN = "#6FE3A1"
RED = "#FF7777"
ROAD = "#344753"
CITY_POS = {
    "Company 1": (0.30, 0.25),
    "Company 2": (0.70, 0.25),
    "Company 3": (0.30, 0.75),
    "Company 4": (0.70, 0.75),

    "Bank": (0.15, 0.15),
    "Hospital": (0.40, 0.10),
    "Store": (0.60, 0.10),
    "Telecom": (0.85, 0.15),
    "University": (0.90, 0.40),
    "Insurance": (0.85, 0.65),
    "Airline": (0.70, 0.90),
    "Factory": (0.50, 0.90),
    "Ministry": (0.25, 0.85),
    "Media": (0.10, 0.55)
}

def balance_problem(demand):
    cost = COST.copy()
    supply = list(SUPPLY)
    demand = list(demand)
    rows, cols = CITIES.copy(), DEPOTS.copy()
    total_s, total_d = sum(supply), sum(demand)
    if total_s < total_d:
        gap = total_d - total_s
        cost = np.vstack((cost, np.zeros((1, cost.shape[1]))))
        supply.append(gap)
        rows.append("Shortage")
    elif total_s > total_d:
        gap = total_s - total_d
        cost = np.hstack((cost, np.zeros((cost.shape[0], 1))))
        demand.append(gap)
        cols.append("Surplus")
    return cost, supply, demand, rows, cols, total_s, total_d

def vogel(cost, supply, demand):
    c = cost.copy()
    s, d = list(supply), list(demand)
    alloc = np.zeros_like(c)
    while any(x > 0 for x in s) and any(x > 0 for x in d):
        candidates = []
        for i in range(len(s)):
            if s[i] <= 0: continue
            vals = sorted(c[i, j] for j in range(len(d)) if d[j] > 0 and np.isfinite(c[i,j]))
            if vals:
                penalty = vals[1] - vals[0] if len(vals) > 1 else vals[0]
                candidates.append((penalty, "r", i))
        for j in range(len(d)):
            if d[j] <= 0: continue
            vals = sorted(c[i, j] for i in range(len(s)) if s[i] > 0 and np.isfinite(c[i,j]))
            if vals:
                penalty = vals[1] - vals[0] if len(vals) > 1 else vals[0]
                candidates.append((penalty, "c", j))
        if not candidates: break
        _, typ, idx = max(candidates, key=lambda x: (x[0], x[1] == "r"))
        if typ == "r":
            i = idx
            j = min((j for j in range(len(d)) if d[j] > 0), key=lambda j: c[i,j])
        else:
            j = idx
            i = min((i for i in range(len(s)) if s[i] > 0), key=lambda i: c[i,j])
        q = min(s[i], d[j])
        alloc[i,j] += q
        s[i] -= q; d[j] -= q
        if s[i] == 0: c[i,:] = np.inf
        if d[j] == 0: c[:,j] = np.inf
    return alloc, float(np.sum(alloc * np.nan_to_num(cost, nan=0)))

def least_cost(cost, supply, demand):
    c = cost.copy()
    s, d = list(supply), list(demand)
    alloc = np.zeros_like(c)
    while any(x > 0 for x in s) and any(x > 0 for x in d):
        options = [(c[i,j], i, j) for i in range(len(s)) for j in range(len(d))
                   if s[i] > 0 and d[j] > 0 and np.isfinite(c[i,j])]
        if not options: break
        _, i, j = min(options)
        q = min(s[i], d[j])
        alloc[i,j] += q
        s[i] -= q; d[j] -= q
        if s[i] == 0: c[i,:] = np.inf
        if d[j] == 0: c[:,j] = np.inf
    return alloc, float(np.sum(alloc * np.nan_to_num(cost, nan=0)))

def initial_basis(alloc, cost):
    m, n = cost.shape
    basis = {(i,j) for i in range(m) for j in range(n) if alloc[i,j] > 0}
    parent = list(range(m+n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a,b):
        parent[find(a)] = find(b)
    for i,j in basis: union(i, m+j)
    for _,i,j in sorted((cost[i,j],i,j) for i in range(m) for j in range(n)):
        if len(basis) >= m+n-1: break
        if find(i) != find(m+j):
            basis.add((i,j)); union(i,m+j)
    return basis

def find_cycle(basis, start, m):
    # Find a closed alternating row/column cycle containing the entering cell.
    cells = set(basis)
    cells.add(start)
    row_to_cols, col_to_rows = {}, {}
    for i,j in cells:
        row_to_cols.setdefault(i, []).append(j)
        col_to_rows.setdefault(j, []).append(i)
    def dfs(cell, horizontal, path):
        i,j = cell
        choices = ([(i, jj) for jj in row_to_cols.get(i, []) if jj != j]
                   if horizontal else [(ii, j) for ii in col_to_rows.get(j, []) if ii != i])
        for nxt in choices:
            if nxt == start and len(path) >= 4:
                return path + [start]
            if nxt in path or nxt == start: continue
            result = dfs(nxt, not horizontal, path + [nxt])
            if result: return result
        return None
    return dfs(start, True, [start]) or dfs(start, False, [start])

def modi(cost, start_alloc):
    alloc = start_alloc.copy()
    m,n = cost.shape
    for _ in range(500):
        basis = initial_basis(alloc, cost)
        u, v = [None]*m, [None]*n
        u[0] = 0
        changed = True
        while changed:
            changed = False
            for i,j in basis:
                if u[i] is not None and v[j] is None:
                    v[j] = cost[i,j] - u[i]; changed = True
                elif v[j] is not None and u[i] is None:
                    u[i] = cost[i,j] - v[j]; changed = True
        u = [0 if x is None else x for x in u]
        v = [0 if x is None else x for x in v]
        best = None
        for i in range(m):
            for j in range(n):
                if (i,j) not in basis:
                    delta = cost[i,j] - u[i] - v[j]
                    if delta < -1e-8 and (best is None or delta < best[0]):
                        best = (delta,i,j)
        if best is None:
            return alloc, float(np.sum(alloc * cost))
        _, i, j = best
        cycle = find_cycle(basis, (i,j), m)
        if not cycle or len(cycle) < 5:
            # Keep the best feasible initial solution rather than corrupting allocation.
            return alloc, float(np.sum(alloc * cost))
        cycle = cycle[:-1]
        minus = cycle[1::2]
        theta = min(alloc[a,b] for a,b in minus)
        alloc[i,j] += theta
        for a,b in cycle[2::2]: alloc[a,b] += theta
        for a,b in minus: alloc[a,b] -= theta
        alloc[np.abs(alloc) < 1e-8] = 0
    return alloc, float(np.sum(alloc * cost))

def solve_scenario(demand):
    c,s,d,rows,cols,ts,td = balance_problem(demand)
    va, vc = vogel(c,s,d)
    la, lc = least_cost(c,s,d)
    oa, oc = modi(c,va)
    return {"cost":c,"supply":s,"demand":d,"rows":rows,"cols":cols,
            "total_supply":ts,"total_demand":td,"vam":vc,"least":lc,
            "modi":oc,"allocation":oa,"vam_alloc":va,"least_alloc":la}

class TransportApp:
    def __init__(self, root):
        self.root = root
        root.title("RouteFlow | Transportation Operations Simulator")
        root.geometry("1380x850")
        root.minsize(1120, 720)
        root.configure(bg=BG)
        self.current = 0
        self.results = [solve_scenario(x[1]) for x in SCENARIOS]
        self.animating = False
        self.animation_after = None
        self.car_phase = 0
        self.active_method = "MODI"
        self.build_ui()
        self.show_scenario(0)

    def build_ui(self):
        header = tk.Frame(self.root, bg=BG)
        header.pack(fill="x", padx=24, pady=(18,12))
        brand = tk.Frame(header, bg=BG)
        brand.pack(side="left")
        tk.Label(brand, text="ROUTE", font=("Segoe UI", 23, "bold"), fg=TEXT, bg=BG).pack(side="left")
        tk.Label(brand, text="FLOW", font=("Segoe UI", 23, "bold"), fg=CYAN, bg=BG).pack(side="left")
        tk.Label(brand, text="   TRANSPORTATION OPERATIONS SIMULATOR", font=("Segoe UI", 10, "bold"),
                 fg=MUTED, bg=BG).pack(side="left", pady=(10,0))
        self.status = tk.Label(header, text="● SYSTEM READY", font=("Segoe UI", 10, "bold"),
                               fg=GREEN, bg=BG)
        self.status.pack(side="right", pady=12)

        tabs = tk.Frame(self.root, bg=BG)
        tabs.pack(fill="x", padx=24, pady=(0,12))
        self.tab_buttons = []
        for i,(title,_,_) in enumerate(SCENARIOS):
            b = tk.Button(tabs, text=f"0{i+1}  {title}", command=lambda x=i:self.show_scenario(x),
                          font=("Segoe UI", 10, "bold"), bd=0, padx=18, pady=12,
                          bg=PANEL2, fg=TEXT, activebackground=CYAN, activeforeground=BG,
                          cursor="hand2")
            b.pack(side="left", padx=(0,8))
            self.tab_buttons.append(b)

        body = tk.Frame(self.root, bg=BG)
        body.pack(fill="both", expand=True, padx=24, pady=(0,18))
        left = tk.Frame(body, bg=PANEL, highlightbackground="#2D414D", highlightthickness=1)
        left.pack(side="left", fill="both", expand=True)
        top = tk.Frame(left, bg=PANEL)
        top.pack(fill="x", padx=18, pady=(15,5))
        self.scene_title = tk.Label(top, text="", font=("Segoe UI", 15, "bold"), fg=TEXT, bg=PANEL)
        self.scene_title.pack(side="left")
        self.live_label = tk.Label(top, text="LIVE NETWORK", font=("Segoe UI", 9, "bold"), fg=CYAN, bg=PANEL)
        self.live_label.pack(side="right")
        self.canvas = tk.Canvas(left, bg="#14232C", highlightthickness=0)
        self.canvas.pack(fill="both", expand=True, padx=12, pady=(4,12))
        self.canvas.bind("<Configure>", lambda e:self.draw_scene())

        right = tk.Frame(body, bg=BG, width=340)
        right.pack(side="right", fill="y", padx=(14,0))
        right.pack_propagate(False)
        self.build_metrics(right)
        self.build_controls(right)

        footer = tk.Frame(self.root, bg=BG)
        footer.pack(fill="x", padx=24, pady=(0,14))
        self.footer_text = tk.Label(footer, text="", font=("Segoe UI", 10), fg=MUTED, bg=BG, anchor="w")
        self.footer_text.pack(fill="x")

    def build_metrics(self, parent):
        tk.Label(parent, text="OPERATIONS OVERVIEW", font=("Segoe UI", 10, "bold"),
                 fg=MUTED, bg=BG).pack(anchor="w", pady=(0,8))
        self.metric_cost = self.metric_card(parent, "OPTIMIZED TRANSPORT COST", "—", CYAN)
        self.metric_units = self.metric_card(parent, "UNITS IN TRANSIT", "—", ORANGE)
        self.metric_balance = self.metric_card(parent, "SUPPLY / DEMAND", "—", GREEN)
        tk.Label(parent, text="ROUTE OPTIMIZER", font=("Segoe UI", 10, "bold"),
                 fg=MUTED, bg=BG).pack(anchor="w", pady=(16,8))
        method_frame = tk.Frame(parent, bg=PANEL)
        method_frame.pack(fill="x")
        self.method_buttons = {}
        for method in ("VAM","Least Cost","MODI"):
            b = tk.Button(method_frame, text=method, command=lambda m=method:self.set_method(m),
                          font=("Segoe UI", 9, "bold"), bd=0, padx=8, pady=9,
                          bg=PANEL2, fg=TEXT, activebackground=CYAN, cursor="hand2")
            b.pack(fill="x", padx=10, pady=4)
            self.method_buttons[method] = b
        self.method_info = tk.Label(parent, text="", font=("Segoe UI", 9), fg=MUTED,
                                    bg=BG, justify="left", wraplength=315)
        self.method_info.pack(fill="x", pady=(9,0))

    def metric_card(self, parent, title, value, color):
        card = tk.Frame(parent, bg=PANEL, highlightbackground="#2D414D", highlightthickness=1)
        card.pack(fill="x", pady=4)
        tk.Label(card, text=title, font=("Segoe UI", 8, "bold"), fg=MUTED, bg=PANEL).pack(anchor="w", padx=12, pady=(10,2))
        val = tk.Label(card, text=value, font=("Segoe UI", 19, "bold"), fg=color, bg=PANEL)
        val.pack(anchor="w", padx=12, pady=(0,10))
        return val

    def build_controls(self, parent):
        tk.Label(parent, text="SIMULATION CONTROLS", font=("Segoe UI", 10, "bold"),
                 fg=MUTED, bg=BG).pack(anchor="w", pady=(16,8))
        self.play_btn = tk.Button(parent, text="▶  START VEHICLE FLOW", command=self.toggle_animation,
                                  font=("Segoe UI", 10, "bold"), bd=0, pady=12,
                                  bg=CYAN, fg=BG, activebackground="#79F0E8", cursor="hand2")
        self.play_btn.pack(fill="x")
        tk.Button(parent, text="↻  RESET VIEW", command=self.reset_view,
                  font=("Segoe UI", 10, "bold"), bd=0, pady=10,
                  bg=PANEL2, fg=TEXT, activebackground=CYAN, cursor="hand2").pack(fill="x", pady=(7,0))
        self.route_label = tk.Label(parent, text="", font=("Segoe UI", 9), fg=TEXT,
                                    bg=PANEL, justify="left", anchor="nw", padx=12, pady=12,
                                    wraplength=310)
        self.route_label.pack(fill="both", expand=True, pady=(12,0))

    def show_scenario(self, idx):
        self.current = idx
        for i,b in enumerate(self.tab_buttons):
            b.configure(bg=CYAN if i == idx else PANEL2, fg=BG if i == idx else TEXT)
        title, demand, desc = SCENARIOS[idx]
        result = self.results[idx]
        self.scene_title.configure(text=title.title())
        self.metric_cost.configure(text=f"{result['modi']:,.0f} EGP")
        self.metric_units.configure(text=f"{int(np.sum(result['allocation']))} units")
        self.metric_balance.configure(text=f"{result['total_supply']} / {result['total_demand']}")
        self.footer_text.configure(text=desc)
        self.set_method(self.active_method)
        self.draw_scene()

    def set_method(self, method):
        self.active_method = method
        for m,b in self.method_buttons.items():
            b.configure(bg=CYAN if m == method else PANEL2, fg=BG if m == method else TEXT)
        result = self.results[self.current]
        costs = {"VAM":result["vam"], "Least Cost":result["least"], "MODI":result["modi"]}
        self.metric_cost.configure(text=f"{costs[method]:,.0f} EGP")
        descriptions = {
            "VAM":"Vogel's Approximation Method builds an efficient starting plan using row/column penalties.",
            "Least Cost":"Ships units through the currently cheapest available route first.",
            "MODI":"Improves the initial plan toward a lower-cost transportation solution."
        }
        self.method_info.configure(text=descriptions[method])
        self.draw_scene()

    def get_alloc(self):
        r = self.results[self.current]
        return r["vam_alloc"] if self.active_method == "VAM" else r["least_alloc"] if self.active_method == "Least Cost" else r["allocation"]

    def draw_scene(self):
        if not hasattr(self, "canvas"): return
        c = self.canvas
        c.delete("all")
        w = max(c.winfo_width(), 700); h = max(c.winfo_height(), 480)
        # Subtle grid and decorative terrain
        for x in range(0,w,38):
            c.create_line(x,0,x,h,fill="#1A2B35")
        for y in range(0,h,38):
            c.create_line(0,y,w,y,fill="#1A2B35")
        c.create_oval(w*.31,h*.05,w*.73,h*.95,fill="#172B31",outline="")
        c.create_text(w*.52,h*.91,text="NILE CORRIDOR  •  EGYPT TRANSPORT NETWORK",
                      fill="#52717B",font=("Segoe UI",8,"bold"))
        # Main road spine and branches
        spine = [(w*.49,h*.24),(w*.50,h*.34),(w*.53,h*.45),(w*.57,h*.57),
                 (w*.59,h*.70),(w*.57,h*.83)]
        c.create_line(*[v for p in spine for v in p], fill=ROAD, width=20, smooth=True, capstyle="round")
        c.create_line(*[v for p in spine for v in p], fill="#607782", width=2, smooth=True, dash=(8,7))
        for city in CITIES:
            x,y = CITY_POS[city]
            px,py = x*w,y*h
            c.create_line(px,py, w*.5,py, fill="#263B46", width=7)
            c.create_line(px,py, w*.5,py, fill="#526A75", width=1, dash=(5,5))
        # Active allocation routes
        alloc = self.get_alloc()
        rows, cols = self.results[self.current]["rows"], self.results[self.current]["cols"]
        maxval = max(1, float(np.max(alloc)))
        routes = []
        for i,row in enumerate(rows):
            if row not in CITY_POS: continue
            for j,col in enumerate(cols):
                q = alloc[i,j]
                if q <= 0 or col not in CITY_POS: continue
                x1,y1 = CITY_POS[row]; x2,y2 = CITY_POS[col]
                p1=(x1*w,y1*h); p2=(x2*w,y2*h)
                midx=(p1[0]+p2[0])/2; midy=(p1[1]+p2[1])/2
                curve = (midx, midy - min(45,abs(p2[0]-p1[0])*.12))
                color = CYAN if q >= maxval*.55 else "#638A93"
                width = 1.5 + 4.0*q/maxval
                c.create_line(*p1,*curve,*p2,fill=color,width=width,smooth=True, stipple="gray50")
                routes.append((p1,curve,p2,q,color,row,col))
        # Vehicles moving along selected allocation routes
        if self.animating and routes:
            for k, (p1,pm,p2,q,color,row,col) in enumerate(routes):
                t = (self.car_phase*.012 + k*.173) % 1.0
                if t < .5:
                    z=t*2; x=p1[0]+(pm[0]-p1[0])*z; y=p1[1]+(pm[1]-p1[1])*z
                else:
                    z=(t-.5)*2; x=pm[0]+(p2[0]-pm[0])*z; y=pm[1]+(p2[1]-pm[1])*z
                self.draw_car(c,x,y,color, 1.0 if q >= maxval*.5 else .78)
        # City and depot markers, labels
        for name in CITIES + DEPOTS:
            if name not in CITY_POS: continue
            x,y=CITY_POS[name]; px,py=x*w,y*h
            is_city = name in CITIES
            r=8 if is_city else 5
            c.create_oval(px-r-3,py-r-3,px+r+3,py+r+3,fill="#203E48",outline="")
            c.create_oval(px-r,py-r,px+r,py+r,fill=ORANGE if is_city else CYAN,outline=TEXT,width=1)
            dx,dy = (10,-14) if name in ("Cairo","Alexandria","Aswan","Luxor") else (7,9)
            c.create_text(px+dx,py+dy,text=name,fill=TEXT if is_city else "#B5C8D0",
                          font=("Segoe UI",9 if is_city else 8,"bold" if is_city else "normal"),anchor="w")
        # Legend
        c.create_rectangle(15,15,220,83,fill="#172630",outline="#344A55")
        c.create_oval(27,28,38,39,fill=ORANGE,outline=TEXT)
        c.create_text(47,34,text="Supply city",fill=TEXT,font=("Segoe UI",9),anchor="w")
        c.create_oval(27,48,38,59,fill=CYAN,outline=TEXT)
        c.create_text(47,54,text="Depot / destination",fill=TEXT,font=("Segoe UI",9),anchor="w")
        c.create_line(27,71,39,71,fill=CYAN,width=4)
        c.create_text(47,71,text="Allocated transport route",fill=TEXT,font=("Segoe UI",9),anchor="w")
        # Right-side route dispatch mini panel
        self.route_label.configure(text=self.route_summary())

    def draw_car(self, canvas, x, y, color, scale=1):
        # Small top-down delivery van: body + cabin + wheels
        ww, hh = 18*scale, 10*scale
        canvas.create_oval(x-ww/2-2,y-hh/2-2,x+ww/2+2,y+hh/2+2,fill="#0B141A",outline="")
        canvas.create_rectangle(x-ww/2,y-hh/2,x+ww/2,y+hh/2,fill=color,outline="#EAF7F8",width=1)
        canvas.create_rectangle(x-ww*.12,y-hh*.34,x+ww*.27,y+hh*.34,fill="#142B36",outline="")
        canvas.create_line(x-ww*.35,y-hh*.25,x-ww*.35,y+hh*.25,fill="#FFFFFF",width=1)

    def route_summary(self):
        alloc = self.get_alloc()
        r = self.results[self.current]
        lines = ["ACTIVE DISPATCH ROUTES", "────────────────────────"]
        found = 0
        for i,row in enumerate(r["rows"]):
            for j,col in enumerate(r["cols"]):
                q = alloc[i,j]
                if q > 0 and row in CITIES and col in DEPOTS:
                    lines.append(f"▸ {row}  →  {col}\n   {int(q)} units  ·  {q*r['cost'][i,j]:,.0f} EGP")
                    found += 1
                    if found >= 5:
                        break
            if found >= 5: break
        if not found: lines.append("No real-city dispatches in this view.")
        lines.append("")
        lines.append(f"Routes displayed: {sum(1 for i,row in enumerate(r['rows']) for j,col in enumerate(r['cols']) if alloc[i,j] > 0 and row in CITIES and col in DEPOTS)}")
        return "\n".join(lines)

    def toggle_animation(self):
        self.animating = not self.animating
        self.play_btn.configure(text="Ⅱ  PAUSE VEHICLE FLOW" if self.animating else "▶  START VEHICLE FLOW",
                                bg=ORANGE if self.animating else CYAN)
        self.status.configure(text="● VEHICLES IN MOTION" if self.animating else "● SYSTEM READY",
                              fg=ORANGE if self.animating else GREEN)
        if self.animating: self.animate()
        elif self.animation_after:
            self.root.after_cancel(self.animation_after)
            self.animation_after = None
        self.draw_scene()

    def animate(self):
        if not self.animating: return
        self.car_phase += 1
        self.draw_scene()
        self.animation_after = self.root.after(70, self.animate)

    def reset_view(self):
        self.animating = False
        if self.animation_after:
            self.root.after_cancel(self.animation_after)
            self.animation_after = None
        self.car_phase = 0
        self.play_btn.configure(text="▶  START VEHICLE FLOW", bg=CYAN)
        self.status.configure(text="● SYSTEM READY", fg=GREEN)
        self.draw_scene()

if __name__ == "__main__":
    root = tk.Tk()
    app = TransportApp(root)
    root.mainloop()
